# Get DMPlex solver to work

## Imports

In [1]:
# | code-fold: true
# | code-summary: "Load packages"
# | output: false

import os
import numpy as np

from zoomy_core.model.models.shallow_water import ShallowWaterEquations as SWE
from zoomy_core.model.models.shallow_water_topo import ShallowWaterEquationsWithTopo as SWEB

import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
from zoomy_core.misc.misc import Zstruct, Settings
from zoomy_core.transformation.to_c import CppModel, CppNumerics
import zoomy_core.misc.misc as misc
import zoomy_core.mesh.petscmesh as petscMesh


## Model

In [2]:
settings = Settings(
    name="SME",
    output=Zstruct(
        directory=f"outputs/dmplex", filename="dmplex", output_snapshots=2
    ),
)

2026-01-20 08:40:22.046 | WARNING  | zoomy_core.misc.misc:__init__:199 - No 'clean_directory' attribute found in output Zstruct. Default: 'False'
2026-01-20 08:40:22.047 | WARNING  | zoomy_core.misc.misc:__init__:199 - No 'snapshots' attribute found in output Zstruct. Default: '2'


In [3]:
bcs = BC.BoundaryConditions(
    [
        BC.Extrapolation(tag="inflow"),
        BC.Extrapolation(tag="wall"),
    ]
)

def custom_ic(x):
    Q = np.zeros(4, dtype=float)
    Q[1] = 0.01
    return Q

ic = IC.UserFunction(custom_ic)

model = SWEB(
    boundary_conditions=bcs,
    initial_conditions=ic,
    aux_variables=1,
)

main_dir = misc.get_main_directory()
mesh = petscMesh.PetscMesh.from_gmsh(
    os.path.join(main_dir, "meshes/channel_quad_2d/mesh_coarse.msh")
)


In [4]:
mesh.to_h5(os.path.join(main_dir, settings.output.directory, 'ic.h5'))